In [1]:
%load_ext autoreload
%autoreload 2
%load_ext dotenv
%dotenv

In [2]:
import os

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"  # see issue #152
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [3]:
### Shortcut for package import
from pkgimp import *

from nb2p import database, config
from nb2p.notebook import Notebook
from nb2p import astparse
from nb2p.dfgtree import DFGTree, preprocess
import materialize

/home/haotian/r/ascent/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
# Load model directly
from tokenizers import Tokenizer
from transformers import RobertaTokenizer, RobertaModel, RobertaConfig, RobertaForSequenceClassification
# from model import Model
from compressor.model import Model

In [6]:
torch.set_num_threads(1)
torch.get_num_threads()

1

In [9]:
def get_xs_model(model_dir: str, size: int):
    config = RobertaConfig.from_pretrained("microsoft/graphcodebert-base")
    config.num_attention_heads = 8
    config.hidden_size = 96
    config.intermediate_size = 64
    config.vocab_size = 1000
    config.num_hidden_layers = 12
    config.hidden_dropout_prob = 0.2

    tokenizer_path = os.path.join("compressor", "BPE" + "_" + str(config.vocab_size) + ".json")
    tokenizer = Tokenizer.from_file(tokenizer_path)

    model = Model(RobertaForSequenceClassification(config=config), config, tokenizer)

    model_dir = os.path.join(model_dir, str(size), "model.bin")
    model.load_state_dict(torch.load(model_dir))

    return model, tokenizer

In [10]:
DEVICE = torch.device("cuda")

model, tokenizer = get_xs_model('compressor/GraphCodeBERT/clone_detection/checkpoint', 3)
model.eval()

Model(
  (encoder): RobertaForSequenceClassification(
    (roberta): RobertaModel(
      (embeddings): RobertaEmbeddings(
        (word_embeddings): Embedding(1000, 96, padding_idx=1)
        (position_embeddings): Embedding(514, 96, padding_idx=1)
        (token_type_embeddings): Embedding(1, 96)
        (LayerNorm): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
        (dropout): Dropout(p=0.2, inplace=False)
      )
      (encoder): RobertaEncoder(
        (layer): ModuleList(
          (0-11): 12 x RobertaLayer(
            (attention): RobertaAttention(
              (self): RobertaSelfAttention(
                (query): Linear(in_features=96, out_features=96, bias=True)
                (key): Linear(in_features=96, out_features=96, bias=True)
                (value): Linear(in_features=96, out_features=96, bias=True)
                (dropout): Dropout(p=0.1, inplace=False)
              )
              (output): RobertaSelfOutput(
                (dense): Linear(in_feature

In [11]:
next(model.parameters()).is_cuda

True

In [14]:
from nb2p.dfgtree.dfg import CodeEncodingBuilder

encoding_builder = CodeEncodingBuilder(tokenizer, model)
encoding_builder

<CodeEncodingBuilder device=cuda>

In [15]:
_DEBUG_CODE_STR = """i = 1
from os import (
    path,
    environ
)
import time
def foo():
    if bar:
        baz()
        def qux():
            quux()
def qux(one: str, two, three):
    quux()
qux()"""

inp = encoding_builder.make_input(_DEBUG_CODE_STR)

In [16]:
out = encoding_builder.get_encoding(inp)
out.shape, out

(torch.Size([1, 96]),
 tensor([[ 0.0459,  0.5318,  0.5849,  0.3224, -0.0777,  0.1114, -0.1325,  0.5421,
          -0.2223,  0.5152,  0.2335, -0.3315, -1.0782,  0.3075,  1.2094, -1.3459,
           1.4802,  0.9046,  0.4666, -0.3980, -0.3756,  0.1517, -1.9829, -0.3983,
           0.6725,  1.2005, -1.0401, -0.0304,  0.7750,  0.0961,  0.7896, -0.7715,
          -0.9073,  0.0583, -0.1441, -0.2353, -0.2811, -0.4528,  0.6436, -1.0412,
           0.7175,  0.1609,  2.9971, -0.4830, -0.7275, -0.6534, -0.1935, -0.7905,
           0.4232,  0.1421,  0.7500,  1.1589,  0.4597,  1.3842,  0.2456, -0.6045,
           0.0459, -0.6196, -0.1072, -1.5798, -0.4606, -1.8542, -0.0883,  0.4313,
          -0.1825,  1.7083,  0.2438,  1.4715,  0.5172,  1.4528,  1.0124,  0.7268,
          -0.9141, -1.2062,  0.0710, -0.0667, -0.6395,  0.1283, -0.5220, -0.3555,
          -1.1694,  0.8614,  0.4743, -0.9364, -1.8166, -0.1661,  0.8547,  0.4033,
           1.1883, -1.1638, -1.8919, -0.6588,  0.0107, -1.1938,  0.2290,  0.

In [17]:
### Select dataset to process

DATASET_NAME = "distilkaggle"
# DATASET_NAME = "gh17"
# DATASET_NAME = "pmbf"

In [18]:
DIRS = config.dirs(dataset_name=DATASET_NAME)
DIRS.makedirs()

making dirs: /ssd/haotian/scs/distilkaggle/processed-unixcoder-ast
making dirs: /ssd/haotian/scs/distilkaggle
making dirs: /ssd/haotian/scs/distilkaggle/processed-unixcoder-full
making dirs: /ssd/haotian/scs/distilkaggle/dfgtree
making dirs: /ssd/haotian/scs/distilkaggle/processed-unixcoder-eda
making dirs: /ssd/haotian/scs/distilkaggle/logs/full
making dirs: /ssd/haotian/scs/distilkaggle/models/full
making dirs: /ssd/haotian/scs/distilkaggle/ipynb


In [19]:
parser, lang = astparse.parser()

db, client = database.connect(dataset_name=DATASET_NAME, verbose=True)

Pinged to database nb2p-dk. You successfully connected to MongoDB!


In [20]:
a_notebook_data = database.get_notebooks(db, {"prompted": True, "segments.6": {"$exists": True}}, include_segments=True).next()
print(a_notebook_data['_id'])
a_notebook = Notebook.from_db_result(a_notebook_data)
a_notebook

66f3ce8e975730f2dded4f49


<Notebook(11 segments, 0 markdown cells, 15 code cells)>

## Test embedding

In [21]:
preproc_result = preprocess(a_notebook, parser, lang)
for s in preproc_result['imports']:
    print(s)
    print("___END___")

print("__SEGMENTS__")

for s in preproc_result['segments']:
    print(s)
    print("___END___")

import numpy as np
___END___
import matplotlib.pyplot as plt
___END___
from sklearn.metrics import accuracy_score
___END___
import sklearn
___END___
import sklearn.datasets
___END___
import warnings
___END___
import sklearn.linear_model
___END___
import sklearn.tree
___END___
import sklearn.svm
___END___
from sklearn.linear_model import LinearRegression
___END___
from sklearn.metrics import r2_score
___END___
from sklearn.metrics import mean_squared_error
___END___
from sklearn.preprocessing import PolynomialFeatures
___END___
from sklearn.pipeline import Pipeline
___END___
from sklearn import cross_validation
___END___
import timeit
___END___
__SEGMENTS__
%matplotlib inline
___END___
%matplotlib inline
x = np.linspace(0, 0.5, 100)
plt.plot( x, 0.7 - 0.5*x + 0.3*np.exp(-x*20), label = "Overfitted model")
plt.plot( x, 0.9 - 0.5*x, label = "Non-Overfitted model")
plt.plot( x, 0.6 - 0.5*x, label = "Just Bad model")
axes = plt.gca()
axes.set_ylim([0, 1.1])
plt.legend(loc=3)
plt.suptitle("E

In [22]:
# %%snakeviz
from pprint import pprint

pprint(DFGTree(input=preproc_result, builder=encoding_builder).build())

{'func_defs': [{'code': 'def jitter(X, scale):\n'
                        '    if scale > 0:        \n'
                        '        return X + np.random.normal(0, scale, '
                        'X.shape)\n'
                        '    return X',
                'n_ast_children': 2,
                'name': 'jitter',
                'repr': <DFGNode repr=None children=[<DFGNode repr=(1, 96) children=[] (0))>, <DFGNode repr=(1, 96) children=[] (0))>] (2)>},
               {'code': 'def jitter_test(classifier, X, y, metric_FUNC = '
                        'accuracy_score, sigmas = np.linspace(0, 0.5, 30), '
                        'averaging_N = 5):\n'
                        '    out = []\n'
                        '    for s in sigmas:\n'
                        '        averageAccuracy = 0.0\n'
                        '        for x in range(averaging_N):\n'
                        '            averageAccuracy += metric_FUNC( y, '
                        'classifier.predict(jitt

## List embedding data

In [ ]:
ids = sorted(list(
    map(
        lambda x: x["_id"],
        db.notebooksegments.find(
            {"prompted": True, "segment_ends.3": {"$exists": True}, "n_ast_children_of_segments": {"$lte": 256}},
            {"_id": 1},
        ),
    )
))
len(ids), ids[:3]

In [22]:
X_train_ids, X_test_ids = train_test_split(ids, test_size=0.2, random_state=42)
len(X_train_ids), len(X_test_ids)

(228699, 57175)

In [20]:
OUT_DIR = DIRS.dfgtree
OUT_DIR

PosixPath('/ssd/haotian/scs/pmbf/dfgtree')

## Write to files

In [ ]:
# ⚠️WARNING: This cell includes file writing

ignored = materialize.to_files(
    db, encoding_builder, parser, lang, OUT_DIR, X_test_ids, "test", full_check=True
)

 13%|█████████████▍                                                                                        | 18292/139351 [4:07:49<24:44:21,  1.36it/s, n_ignored=0, fd_len=0, n_seg=5]

In [ ]:
# # ⚠️WARNING: This cell includes file writing

# ignored = materialize.to_files(
#     db, encoding_builder, parser, lang, OUT_DIR, X_train_ids, "train", full_check=True
# )